# 비교 실험: 선행연구 방식 재현 · 평가 설계 효과 분리 · 단순 기준선
최종 노트북(`Animal_Shelter_Analytics_최종_논문용.ipynb`)의 1~9장(데이터 준비·연결·특성·분할·평가 도구)을 그대로 실행한 뒤, 최종 설정(`outputs_final/summary.json`)으로 세 가지 비교 실험을 합니다. 결과는 `outputs_compare/`에 저장하며, 최종 결과 파일은 바꾸지 않습니다. **모든 실험은 보고용이며 최종 모델 선택을 바꾸지 않습니다.**

| 실험 | 질문 |
|---|---|
| E1 선행연구 방식 재현 | 결과 다중분류·정확도·무작위 분할로 평가하면 어떻게 보이며, 안락사 탐지는 어떤가? |
| E2 평가 설계 효과 분리 | 같은 2024년 평가 대상에서, 무작위 학습과 시간 순 학습의 차이는 '분할 방식' 때문인가 '최신 데이터' 때문인가? |
| E3 단순 기준선 | 건강상태·나이 규칙이나 로지스틱 회귀와 비교해 최종 모델이 얼마나 나은가? |

In [1]:
import nbformat, json
_main = nbformat.read("Animal_Shelter_Analytics_최종_논문용.ipynb", as_version=4)
_stop = next(i for i, c in enumerate(_main.cells) if c.cell_type == "markdown" and c.source.startswith("## 10."))
for _c in _main.cells[:_stop]:
    if _c.cell_type == "code":
        exec(_c.source.replace('"figures_final", "outputs_final"', '"figures_compare", "outputs_compare"'), globals())
print("최종 노트북 1~9장 재실행 완료 | 결과 폴더:", OUT_DIR)

,항목,버전
0,python,3.12.7
1,numpy,1.26.4
2,pandas,2.2.2
3,scipy,1.13.1
4,scikit-learn,1.5.1
5,lightgbm,4.6.0
6,xgboost,3.4.1
7,imbalanced-learn,0.12.3
8,shap,0.50.0
9,matplotlib,3.9.2


SEED 42 | B_BOOT 500 | W_DAYS 90 | T_DEPLOY 2024-01-01
입력: Austin_Animal_Center_Intakes_(10_01_2013_to_05_05_2025)_20260925.csv | Austin_Animal_Center_Outcomes_(10_01_2013_to_05_05_2025)_20260925.csv
외부 파일 (Kaggle 2021년판): ..\..\..\..\매트릭스\Animal Shelter 프로젝트\반려동물 데이터셋.zip -> 있음
확인: Austin_Animal_Center_Intakes_(10_01_2013_to_05_05_2025)_20260925.csv (30.1 MB)
확인: Austin_Animal_Center_Outcomes_(10_01_2013_to_05_05_2025)_20260925.csv (25.2 MB)


입소 (173812, 12) | 퇴소 (173775, 12)


,파일,형식,건수,파싱 실패,시각 00:00:00 비율,오프셋 값
0,입소,MM/DD/YYYY hh:mm:ss AM/PM,173812,0,0.0003,-
1,퇴소,ISO 시각 (오프셋 없음),170191,0,0.0012,
2,퇴소,ISO 시각 + UTC 오프셋,3584,0,1.0000,-05:00:3584


오프셋 행의 월별 오프셋 값 (CST -06:00이 나타나는지 확인): {1: '-05:00', 2: '-05:00', 3: '-05:00', 4: '-05:00', 5: '-05:00', 6: '-05:00', 7: '-05:00', 8: '-05:00', 9: '-05:00', 10: '-05:00', 11: '-05:00', 12: '-05:00'}
날짜 전용 판정: 3,584행 (오프셋 행 3,584행 중)
데이터 종료 시점(DATA_END): 2025-05-04 23:44:00


...

나이 결측·오류 삭제: 9행


개·고양이 입소: (163892, 43)


,Wildlife 아님,Wildlife 입소,합계
animal_type,,,
Cat,69313,0,69313
Dog,94587,1,94588
Other,3397,6478,9875
All,167297,6479,173776


,구간,평가 입소,학습 입소 끝,학습 n,평가 n,평가 양성,평가 양성률,평가 라벨 확정 시점 ≤ T_deploy
0,선택 폴드 1,2019-10-04 ~ 2020-10-03,2019-07-05,97558,9924,213,0.0215,True
1,선택 폴드 2,2020-10-03 ~ 2021-10-03,2020-07-04,111405,9629,226,0.0235,True
2,선택 폴드 3,2021-10-03 ~ 2022-10-03,2021-07-04,120449,11295,234,0.0207,True
3,보정 구간,2022-10-03 ~ 2023-10-03,2022-07-04,132997,10197,166,0.0163,True
4,테스트,2024-01-01 ~ 2025-01-01,2022-07-04,132997,10778,238,0.0221,평가용 (해당 없음)
5,보조,2025-01-01 ~ 2025-02-03,2022-07-04,132997,827,24,0.0290,평가용 (해당 없음)


테스트 구간 안의 반복 방문: 429마리가 2회 이상 (907행) -> 군집 부트스트랩 사용
최종 노트북 1~9장 재실행 완료 | 결과 폴더: outputs_compare


In [2]:
S_ = json.load(open(os.path.join("outputs_final", "summary.json"), encoding="utf-8"))
cfg = S_["선택된 설정"]
KIND = {v: k for k, v in FAMILY_LABEL.items()}[cfg["계열"]]
FEATS = cfg["특성"]
FINAL_CAT = [f for f in FEATS if f in BASE_CAT]
FINAL_NUM = [f for f in FEATS if f not in BASE_CAT]
FINAL_IMB, FINAL_PARAMS = cfg["불균형 전략"], cfg["파라미터"]
FINAL_FAMILY = KIND
final_train = train_before(CAL[0])
test_df = block(*TEST)
y_test, g_test = test_df["y_euth"].values, test_df["Animal ID"].values
base = y_test.mean()
print("최종 설정:", cfg["계열"], FINAL_IMB, FINAL_PARAMS)
print("특성:", FEATS)
print(f"최종 학습 {len(final_train):,}건 | 2024 테스트 {len(test_df):,}건 (양성 {int(y_test.sum())}, 기준선 {base:.4f})")


def fit_final(trn):
    return fit_with_strategy(FINAL_IMB, trn, FINAL_CAT, FINAL_NUM, "y_euth", FINAL_PARAMS, family=KIND)


def ap_row(label, p, note="", ref=None):
    b = boot_metric(y_test, [p] + ([ref] if ref is not None else []), g_test, average_precision_score)
    row = {"설정": label, "AP": average_precision_score(y_test, p), "AP 95% CI": ci_str(b[:, 0]),
           "배수": average_precision_score(y_test, p) / base, "ROC-AUC": roc_auc_score(y_test, p), "비고": note}
    if ref is not None:
        d = b[:, 0] - b[:, 1]
        row["기준 대비 AP 차이 [95% CI]"] = f"{d.mean():+.4f} {ci_str(d)}"
    return row

최종 설정: XGBoost (부스팅) 가중치 {'subsample': 1.0, 'n_estimators': 200, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.03, 'colsample_bytree': 0.6}
특성: ['animal_type', 'Intake Type', 'Intake Condition', 'breed_group', 'life_stage', 'Breed', 'Color', 'age_days', 'intake_month', 'is_mixed_breed', 'is_multicolor', 'is_sterilized']
최종 학습 132,997건 | 2024 테스트 10,778건 (양성 238, 기준선 0.0221)


## E1. 선행연구 방식 재현 (결과 다중분류)
선행연구[Kim 외 2021; Parte 2019]처럼 **최종 퇴소 결과 유형**(입양·이송·보호자 반환·안락사·기타)을 다중분류하고 **정확도**로 평가합니다. 모델은 다중 로지스틱 회귀(Kim 외)와 XGBoost(Parte)이며, 특성은 최종 모델과 같은 입소 시점 특성입니다.
- (i) 선행연구 기간(2013-10 ~ 2016-03)을 동물 ID 기준 무작위 80:20
- (ii) 전체 기간(2013-10 ~ 2024-12)을 동물 ID 기준 무작위 80:20
- (iii) 시간 분할: 2022-10-03 이전에 퇴소가 끝난 입소로 학습, 2024년 입소(최종 테스트와 같은 동물)로 평가

정확도와 함께 **안락사 클래스의 Recall·Precision·F1**(최대 확률 클래스 기준)과 **안락사 AP**(확률 순위)를 봅니다.

In [3]:
from sklearn.metrics import accuracy_score, recall_score, precision_recall_fscore_support
from xgboost import XGBClassifier
OUTC = {"Adoption": "입양", "Transfer": "이송", "Return to Owner": "보호자 반환", "Rto-Adopt": "보호자 반환", "Euthanasia": "안락사"}
mc = dc[dc["link_status"] == "연결"].copy()
mc["y_mc"] = mc["Outcome Type"].map(OUTC).fillna("기타")
CLASSES = ["입양", "이송", "보호자 반환", "안락사", "기타"]
mc["y_code"] = mc["y_mc"].map({c: i for i, c in enumerate(CLASSES)})
print(mc["y_mc"].value_counts(normalize=True).round(3).to_dict())


def mc_models():
    return {"다중 로지스틱 회귀": Pipeline([("prep", make_prep(FINAL_CAT, FINAL_NUM, scale=True)), ("model", LogisticRegression(max_iter=3000))]),
            "XGBoost (다중분류)": Pipeline([("prep", make_prep(FINAL_CAT, FINAL_NUM)), ("model", XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=6, subsample=0.8,
                                                                                                         colsample_bytree=0.8, n_jobs=4, random_state=SEED))])}


def eval_mc(setting, tr, te):
    rows = []
    for name, m in mc_models().items():
        m.fit(tr[FEATS], tr["y_code"])
        proba = m.predict_proba(te[FEATS])
        pred = proba.argmax(1)
        y = te["y_code"].values
        e = CLASSES.index("안락사")
        pr, rc, f1, _ = precision_recall_fscore_support(y == e, pred == e, average="binary", zero_division=0)
        rows.append({"설정": setting, "모델": name, "학습 n": len(tr), "평가 n": len(te), "정확도": accuracy_score(y, pred),
                     "매크로 재현율": recall_score(y, pred, average="macro"), "안락사 비율": (y == e).mean(),
                     "안락사 Recall": rc, "안락사 Precision": pr, "안락사 F1": f1, "안락사 AP": average_precision_score(y == e, proba[:, e])})
    return rows


rows = []
k1 = mc[mc["t_in"] < pd.Timestamp("2016-04-01")]
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
a, b = next(gss.split(k1, groups=k1["Animal ID"]))
rows += eval_mc("(i) 2013-10~2016-03 무작위 80:20", k1.iloc[a], k1.iloc[b])
k2 = mc[mc["t_in"] < pd.Timestamp("2025-01-01")]
a, b = next(gss.split(k2, groups=k2["Animal ID"]))
rows += eval_mc("(ii) 2013-10~2024-12 무작위 80:20", k2.iloc[a], k2.iloc[b])
tr3 = mc[mc["t_out"] < CAL[0]]
te3 = mc[mc.index.isin(test_df.index)]
rows += eval_mc("(iii) 시간 분할 (학습 ~2022-10, 평가 2024)", tr3, te3)
e1 = pd.DataFrame(rows)
display(e1.round(3))
save_table(e1, "E1_prior_study_replication.csv")

{'입양': 0.508, '이송': 0.288, '보호자 반환': 0.164, '안락사': 0.028, '기타': 0.011}


,설정,모델,학습 n,평가 n,정확도,매크로 재현율,안락사 비율,안락사 Recall,안락사 Precision,안락사 F1,안락사 AP
0,(i) 2013-10~2016-03 무작위 80:20,다중 로지스틱 회귀,33926,8475,0.565,0.391,0.052,0.190,0.392,0.256,0.272
1,(i) 2013-10~2016-03 무작위 80:20,XGBoost (다중분류),33926,8475,0.615,0.436,0.052,0.286,0.460,0.353,0.323
2,(ii) 2013-10~2024-12 무작위 80:20,다중 로지스틱 회귀,127743,31939,0.599,0.341,0.027,0.074,0.276,0.117,0.174
3,(ii) 2013-10~2024-12 무작위 80:20,XGBoost (다중분류),127743,31939,0.641,0.391,0.027,0.129,0.380,0.192,0.218
4,"(iii) 시간 분할 (학습 ~2022-10, 평가 2024)",다중 로지스틱 회귀,134944,10651,0.670,0.353,0.023,0.057,0.161,0.084,0.105
5,"(iii) 시간 분할 (학습 ~2022-10, 평가 2024)",XGBoost (다중분류),134944,10651,0.675,0.389,0.023,0.036,0.155,0.059,0.110


## E2. 평가 설계 효과 분리 (같은 2024년 평가 대상)
평가 대상은 최종 테스트(2024년 입소, 재방문 제외)로 **고정**하고, 학습 데이터만 바꿉니다. 모델 설정은 최종 모델과 같습니다.

| 학습 설정 | 학습 데이터 | 보는 효과 |
|---|---|---|
| T1 시간 순 (주 분석) | 입소+90일 ≤ 2022-10-03 | 기준 |
| T2 시간 순 + 최신 | 입소+90일 ≤ 2024-01-01 | 최신 데이터(약 15개월 추가)의 효과 |
| R1 무작위 (T1과 같은 크기) | 테스트 동물을 뺀 2013~2025-02 관찰 완료 입소에서 무작위 추출, 5회 반복 | 시간 순서를 무시한 학습(미래 포함)의 효과 |
| R2 무작위 (전체) | 위 풀 전체 | 무작위 분할 + 데이터 최대 |

- **T2 - T1** = 최신성 효과, **R1 - T2** = (비슷한 최신성에서) 시간 순서를 무시한 효과에 가깝습니다. 차이는 같은 테스트에서 짝지은 동물 군집 부트스트랩입니다.

In [4]:
T1 = final_train
T2 = lab[lab["t_in"] + W <= T_DEPLOY]
test_ids = set(test_df["Animal ID"])
POOL = lab[~lab["Animal ID"].isin(test_ids)]
comp = []
for name, d in [("T1", T1), ("T2", T2), ("R 풀", POOL)]:
    comp.append({"학습 설정": name, "n": len(d), "양성률": d["y_euth"].mean(), "2023년 이후 입소 비율": (d["t_in"] >= pd.Timestamp("2023-01-01")).mean(),
                 "학습 입소 끝": str(d["t_in"].max().date())})
display(pd.DataFrame(comp).round(4))

p_T1 = fit_final(T1).predict_proba(test_df[FEATS])[:, 1]
p_T2 = fit_final(T2).predict_proba(test_df[FEATS])[:, 1]
r1_preds = []
for sd in range(5):
    r1_preds.append(fit_final(POOL.sample(n=len(T1), random_state=SEED + sd)).predict_proba(test_df[FEATS])[:, 1])
p_R2 = fit_final(POOL).predict_proba(test_df[FEATS])[:, 1]
rows = [ap_row("T1 시간 순 (주 분석)", p_T1, "최종 모델과 같은 학습"),
        ap_row("T2 시간 순 + 최신 (~2023-10)", p_T2, "최신성 효과", ref=p_T1)]
r1_aps = [average_precision_score(y_test, p) for p in r1_preds]
r1_mean = np.mean(r1_preds, axis=0)
row = ap_row("R1 무작위 (T1과 같은 크기, 5회 평균 예측)", r1_mean, f"반복별 AP {min(r1_aps):.3f}~{max(r1_aps):.3f}", ref=p_T2)
row["기준"] = "T2"
rows.append(row)
row = ap_row("R2 무작위 (전체 풀)", p_R2, "무작위 + 최대 데이터", ref=p_T2)
row["기준"] = "T2"
rows.append(row)
for r_ in rows[:2]:
    r_.setdefault("기준", "T1")
e2 = pd.DataFrame(rows)
display(e2.round(4))
save_table(e2, "E2_evaluation_design.csv")
# 참고: 같은 학습(T1)으로 평가 기간만 바꾸면? -> 최종 노트북 12-1절(무작위 분할 전체 평가 vs 2024 입소만)과 함께 해석

,학습 설정,n,양성률,2023년 이후 입소 비율,학습 입소 끝
0,T1,132997,0.0284,0.0000,2022-07-04
1,T2,146630,0.0274,0.0575,2023-10-02
2,R 풀,150196,0.0272,0.0798,2025-02-03


,설정,AP,AP 95% CI,배수,ROC-AUC,비고,기준,기준 대비 AP 차이 [95% CI]
0,T1 시간 순 (주 분석),0.1077,"[0.087, 0.143]",4.8759,0.8109,최종 모델과 같은 학습,T1,NaN
1,T2 시간 순 + 최신 (~2023-10),0.1084,"[0.087, 0.145]",4.9084,0.8143,최신성 효과,T1,"+0.0008 [-0.004, 0.005]"
2,"R1 무작위 (T1과 같은 크기, 5회 평균 예측)",0.1076,"[0.087, 0.143]",4.8740,0.8147,반복별 AP 0.108~0.109,T2,"-0.0009 [-0.004, 0.001]"
3,R2 무작위 (전체 풀),0.1076,"[0.087, 0.143]",4.8735,0.8149,무작위 + 최대 데이터,T2,"-0.0009 [-0.004, 0.002]"


## E3. 단순 기준선과 비교 (2024 테스트)
- **무작위 선택**: AP = 양성 비율.
- **규칙 기반**: 건강상태가 부상·질병이면 먼저, 그 안에서 나이가 많은 순 (학습 없음, 미리 정한 규칙).
- **로지스틱 회귀**: 같은 특성, 클래스 가중치, 같은 학습 데이터(T1).
- **최종 모델**: XGBoost (주 분석).
같은 검토량으로 비교하기 위해, 최종 모델이 보정 구간 임곗값으로 선별한 건수만큼 각 방법의 상위 동물을 검토한다고 보고 Recall·Precision을 계산합니다.

In [5]:
cond = test_df["Intake Condition"].fillna("")
rule = 10 * cond.isin(["Injured", "Sick"]).astype(float) + 5 * (~cond.isin(["Normal", "Nursing", "Neonatal", "Injured", "Sick", ""])).astype(float) + test_df["age_days"].fillna(0) / 3650
lr = make_model("LR", FINAL_CAT, FINAL_NUM, weighted=True, y=T1["y_euth"]).fit(T1[FEATS], T1["y_euth"])
p_lr = lr.predict_proba(test_df[FEATS])[:, 1]
m_final = fit_final(T1)
p_cal = m_final.predict_proba(cal_df[FEATS])[:, 1]


def thr_for_recall(y, p, target=0.8):
    prec, rec, thr = precision_recall_curve(y, p)
    ok = np.where(rec[:-1] >= target)[0]
    return thr[ok[np.argmax(prec[:-1][ok])]]


k_flag = int((p_T1 >= thr_for_recall(cal_df["y_euth"].values, p_cal)).sum())


def at_k(p):
    idx = np.argsort(-np.asarray(p), kind="mergesort")[:k_flag]
    tp = y_test[idx].sum()
    return tp / y_test.sum(), tp / k_flag


rows = []
rng = np.random.default_rng(SEED)
for name, p in [("무작위 선택", rng.random(len(y_test))), ("규칙 기반 (건강상태→나이)", rule.values), ("로지스틱 회귀", p_lr), ("최종 모델 (XGBoost)", p_T1)]:
    r_ = ap_row(name, np.asarray(p), ref=p_T1 if name != "최종 모델 (XGBoost)" else None)
    rec_, pre_ = at_k(p)
    r_.update({"같은 검토량 Recall": rec_, "같은 검토량 Precision": pre_, "검토 건수": k_flag})
    rows.append(r_)
e3 = pd.DataFrame(rows)
display(e3.round(4))
save_table(e3, "E3_baselines.csv")

,설정,AP,AP 95% CI,배수,ROC-AUC,비고,기준 대비 AP 차이 [95% CI],같은 검토량 Recall,같은 검토량 Precision,검토 건수
0,무작위 선택,0.0241,"[0.020, 0.034]",1.0897,0.5078,,"-0.0867 [-0.117, -0.061]",0.3319,0.0226,3492
1,규칙 기반 (건강상태→나이),0.0907,"[0.071, 0.121]",4.1060,0.7725,,"-0.0181 [-0.041, 0.001]",0.7563,0.0515,3492
2,로지스틱 회귀,0.1023,"[0.082, 0.135]",4.6317,0.8031,,"-0.0055 [-0.023, 0.013]",0.7773,0.0530,3492
3,최종 모델 (XGBoost),0.1077,"[0.087, 0.143]",4.8759,0.8109,,NaN,0.7815,0.0533,3492


## E1 보강. 다중분류 모델을 안락사 확률 순위로 쓰면 (보고용)
E1 (iii) 시간 분할의 다중분류 모델(XGBoost, 다중 로지스틱 회귀)을 **최대 확률 클래스 판정 대신 안락사 클래스 확률 순위**로 쓰고, E3과 같은 검토량(`k_flag`)만큼 상위 입소를 검토한다고 볼 때의 안락사 재현율·정밀도입니다. 낮은 안락사 탐지율이 모델 때문인지, 판정·보고 방식 때문인지를 봅니다. **보고용이며 어떤 선택에도 쓰지 않습니다.** 기존 E1 표는 바꾸지 않습니다.
- E1 (iii)의 평가 대상(`te3`, 연결된 2024 입소, 최종 결과 라벨)은 E3의 테스트(`test_df`, 미연결 입소 포함, 90일 라벨)와 다릅니다. 그래서 **(a) `te3`에서 최종 결과 라벨**, **(b) `te3`와 `test_df`에 공통인 입소에서 90일 라벨** 두 가지로 계산합니다.
- 검토 건수는 E3 검토량 그대로인 값과, 각 대상의 입소 수 비율(대상 n ÷ 테스트 n)로 환산한 값을 함께 둡니다. (b)에는 같은 대상에서의 최종 이진 모델 값을 참고로 둡니다.

In [6]:
# E1 보강 (보고용): E1 (iii)과 같은 설정으로 다시 학습해 안락사 클래스 확률을 얻고, 확률 순 상위 k건의 재현율·정밀도를 계산
e_ = CLASSES.index("안락사")
SET3 = e1["설정"].unique()[2]
e1_iii = e1[e1["설정"] == SET3].set_index("모델")
p_mc = {}
for name, m in mc_models().items():
    m.fit(tr3[FEATS], tr3["y_code"])
    p_mc[name] = pd.Series(m.predict_proba(te3[FEATS])[:, e_], index=te3.index)
    ap_ = average_precision_score(te3["y_code"].values == e_, p_mc[name].values)
    if abs(ap_ - e1_iii.loc[name, "안락사 AP"]) > 1e-9:
        warnings.warn(f"{name}: 다시 학습한 안락사 AP {ap_:.4f}가 E1 표 {e1_iii.loc[name, '안락사 AP']:.4f}와 다름")


def rank_at_k(y, p, k):
    idx = np.argsort(-np.asarray(p), kind="mergesort")[:k]
    tp = int(np.asarray(y)[idx].sum())
    return tp, tp / np.asarray(y).sum(), tp / k


common = te3.index.intersection(test_df.index)
p_T1_s = pd.Series(p_T1, index=test_df.index)
targets = {"(a) te3 · 최종 결과 라벨": (te3.index, (te3["y_code"].values == e_).astype(int)),
           "(b) te3∩테스트 · 90일 라벨": (common, test_df.loc[common, "y_euth"].values)}
rows = []
for tgt, (ix, yy) in targets.items():
    k_scaled = int(round(k_flag * len(ix) / len(test_df)))
    cands = {name: p.loc[ix].values for name, p in p_mc.items()}
    if tgt.startswith("(b)"):
        cands["최종 모델 (XGBoost 이진, 참고)"] = p_T1_s.loc[ix].values
    for name, pp in cands.items():
        for k_lab, k_ in [(f"E3 검토량 그대로 ({k_flag:,}건)", k_flag), (f"입소 수 비율 환산 ({len(ix):,}/{len(test_df):,})", k_scaled)]:
            tp, rc, pr = rank_at_k(yy, pp, k_)
            rows.append({"평가 대상": tgt, "모델": name, "평가 n": len(ix), "양성 수": int(yy.sum()), "검토 건수 기준": k_lab, "검토 건수": k_,
                         "찾은 양성": tp, "안락사 Recall (확률 순위)": rc, "안락사 Precision (확률 순위)": pr,
                         "참고: 최대 확률 판정 Recall (E1 표)": e1_iii.loc[name, "안락사 Recall"] if (tgt.startswith("(a)") and name in e1_iii.index) else np.nan})
e1r = pd.DataFrame(rows)
display(e1r.round(4))
save_table(e1r, "E1_ranking_recall.csv")

,평가 대상,모델,평가 n,양성 수,검토 건수 기준,검토 건수,찾은 양성,안락사 Recall (확률 순위),안락사 Precision (확률 순위),참고: 최대 확률 판정 Recall (E1 표)
0,(a) te3 · 최종 결과 라벨,다중 로지스틱 회귀,10651,247,"E3 검토량 그대로 (3,492건)",3492,191,0.7733,0.0547,0.0567
1,(a) te3 · 최종 결과 라벨,다중 로지스틱 회귀,10651,247,"입소 수 비율 환산 (10,651/10,778)",3451,191,0.7733,0.0553,0.0567
2,(a) te3 · 최종 결과 라벨,XGBoost (다중분류),10651,247,"E3 검토량 그대로 (3,492건)",3492,204,0.8259,0.0584,0.0364
3,(a) te3 · 최종 결과 라벨,XGBoost (다중분류),10651,247,"입소 수 비율 환산 (10,651/10,778)",3451,203,0.8219,0.0588,0.0364
4,(b) te3∩테스트 · 90일 라벨,다중 로지스틱 회귀,10651,238,"E3 검토량 그대로 (3,492건)",3492,185,0.7773,0.0530,NaN
5,(b) te3∩테스트 · 90일 라벨,다중 로지스틱 회귀,10651,238,"입소 수 비율 환산 (10,651/10,778)",3451,185,0.7773,0.0536,NaN
6,(b) te3∩테스트 · 90일 라벨,XGBoost (다중분류),10651,238,"E3 검토량 그대로 (3,492건)",3492,197,0.8277,0.0564,NaN
7,(b) te3∩테스트 · 90일 라벨,XGBoost (다중분류),10651,238,"입소 수 비율 환산 (10,651/10,778)",3451,196,0.8235,0.0568,NaN
8,(b) te3∩테스트 · 90일 라벨,"최종 모델 (XGBoost 이진, 참고)",10651,238,"E3 검토량 그대로 (3,492건)",3492,188,0.7899,0.0538,NaN
9,(b) te3∩테스트 · 90일 라벨,"최종 모델 (XGBoost 이진, 참고)",10651,238,"입소 수 비율 환산 (10,651/10,778)",3451,187,0.7857,0.0542,NaN


## 요약 (자동 생성)

In [7]:
e1i = e1.set_index(["설정", "모델"])
xi, xii, xiii = [e1i.xs(s, level="설정").loc["XGBoost (다중분류)"] for s in e1["설정"].unique()]
e2i, e3i = e2.set_index("설정"), e3.set_index("설정")
t1, t2 = e2i.iloc[0], e2i.iloc[1]
r1, r2 = e2i.iloc[2], e2i.iloc[3]
# E1 보강 (보고용): 같은 검토량(E3 검토량 그대로)에서 안락사 확률 순위의 재현율
rk_ = e1r[e1r["검토 건수 기준"].str.startswith("E3 검토량")].set_index(["평가 대상", "모델"])["안락사 Recall (확률 순위)"]
ta_, tb_ = list(targets)
rx_, rl_ = rk_[(ta_, "XGBoost (다중분류)")], rk_[(ta_, "다중 로지스틱 회귀")]
ax_, al_ = xiii["안락사 Recall"], e1i.xs(SET3, level="설정").loc["다중 로지스틱 회귀", "안락사 Recall"]
e1r_line = (f"- **E1 보강 (보고용)**: 다중분류 모델도 안락사 확률 순으로 같은 규모({k_flag:,}건)를 검토하면 재현율이 XGBoost {rx_:.3f}·다중 로지스틱 회귀 {rl_:.3f}"
            f"(te3, 최종 결과 라벨), {rk_[(tb_, 'XGBoost (다중분류)')]:.3f}·{rk_[(tb_, '다중 로지스틱 회귀')]:.3f}(공통 입소, 90일 라벨)입니다. "
            f"최대 확률 판정에서는 {ax_:.3f}·{al_:.3f}였으므로"
            + (" 낮은 탐지율은 모델보다 **최대 확률 판정 방식**에서 왔습니다" if min(rx_, rl_) > 2 * max(ax_, al_) else " 판정 방식만으로 차이를 설명하기 어렵습니다")
            + f"(같은 대상의 최종 이진 모델 {rk_[(tb_, '최종 모델 (XGBoost 이진, 참고)')]:.3f}).")
show_md([
    "**E1 선행연구 방식 재현**",
    f"- 선행연구 기간을 무작위로 나누면 XGBoost 정확도 {xi['정확도']:.3f}, 매크로 재현율 {xi['매크로 재현율']:.3f}로 선행연구(정확도 0.627~0.653, 매크로 재현율 0.255)와 비슷한 수준이 재현됩니다.",
    f"- 그러나 안락사 클래스의 Recall은 {xi['안락사 Recall']:.3f}(무작위, 2013~2016), {xii['안락사 Recall']:.3f}(무작위, 전체), {xiii['안락사 Recall']:.3f}(시간 분할)로, "
    "정확도가 높아 보여도 **안락사는 거의 찾지 못합니다**. 최대 확률 클래스로 판정하면 드문 안락사가 다수 클래스에 묻히기 때문입니다.",
    f"- 안락사 AP는 {xi['안락사 AP']:.3f} → {xii['안락사 AP']:.3f} → {xiii['안락사 AP']:.3f}로, 평가 기간·분할이 바뀌면 크게 달라집니다(안락사 비율 {xi['안락사 비율']:.3f}, {xii['안락사 비율']:.3f}, {xiii['안락사 비율']:.3f}).",
    e1r_line,
    "",
    "**E2 평가 설계 효과 분리 (같은 2024 평가 대상)**",
    f"- T1(시간 순, 주 분석) AP {t1['AP']:.3f}, T2(최신 데이터 추가) {t2['AP']:.3f} (T1 대비 {t2['기준 대비 AP 차이 [95% CI]']} → {judge(t2['기준 대비 AP 차이 [95% CI]'])}).",
    f"- R1(무작위, 같은 크기) {r1['AP']:.3f} (T2 대비 {r1['기준 대비 AP 차이 [95% CI]']} → {judge(r1['기준 대비 AP 차이 [95% CI]'])}), "
    f"R2(무작위, 전체) {r2['AP']:.3f} (T2 대비 {r2['기준 대비 AP 차이 [95% CI]']} → {judge(r2['기준 대비 AP 차이 [95% CI]'])}).",
    "- '유의차 없음'은 차이의 95% 구간이 0을 포함한다는 뜻이며, 두 값이 같다는 뜻은 아닙니다.",
    "- 같은 평가 대상에서는 학습 데이터를 시간 순으로 고르든 무작위로 고르든 성능 차이가 작습니다. 최종 노트북 12-1절의 큰 차이(무작위 분할 전체 평가 AP 0.22 vs 시간 검증 0.11)는 "
    "**분할 방식보다 평가 대상(기간·구성)의 차이**에서 온 것으로 해석됩니다. 시간 순 검증의 의의는 성능 수치를 바꾸는 데보다, 선택에 미래 정보가 섞이지 않게 하고 운영 조건의 성능을 제시하는 데 있습니다.",
    "",
    "**E3 단순 기준선**",
    "- " + "; ".join(f"{k} AP {v['AP']:.3f}, 같은 검토량 Recall {v['같은 검토량 Recall']:.3f}" for k, v in e3i.iterrows()) + ".",
    "- 차이 구간은 표의 '기준 대비 AP 차이'를 보세요 (기준 = 최종 모델).",
])

**E1 선행연구 방식 재현**
- 선행연구 기간을 무작위로 나누면 XGBoost 정확도 0.615, 매크로 재현율 0.436로 선행연구(정확도 0.627~0.653, 매크로 재현율 0.255)와 비슷한 수준이 재현됩니다.
- 그러나 안락사 클래스의 Recall은 0.286(무작위, 2013~2016), 0.129(무작위, 전체), 0.036(시간 분할)로, 정확도가 높아 보여도 **안락사는 거의 찾지 못합니다**. 최대 확률 클래스로 판정하면 드문 안락사가 다수 클래스에 묻히기 때문입니다.
- 안락사 AP는 0.323 → 0.218 → 0.110로, 평가 기간·분할이 바뀌면 크게 달라집니다(안락사 비율 0.052, 0.027, 0.023).
- **E1 보강 (보고용)**: 다중분류 모델도 안락사 확률 순으로 같은 규모(3,492건)를 검토하면 재현율이 XGBoost 0.826·다중 로지스틱 회귀 0.773(te3, 최종 결과 라벨), 0.828·0.777(공통 입소, 90일 라벨)입니다. 최대 확률 판정에서는 0.036·0.057였으므로 낮은 탐지율은 모델보다 **최대 확률 판정 방식**에서 왔습니다(같은 대상의 최종 이진 모델 0.790).

**E2 평가 설계 효과 분리 (같은 2024 평가 대상)**
- T1(시간 순, 주 분석) AP 0.108, T2(최신 데이터 추가) 0.108 (T1 대비 +0.0008 [-0.004, 0.005] → 유의차 없음).
- R1(무작위, 같은 크기) 0.108 (T2 대비 -0.0009 [-0.004, 0.001] → 유의차 없음), R2(무작위, 전체) 0.108 (T2 대비 -0.0009 [-0.004, 0.002] → 유의차 없음).
- '유의차 없음'은 차이의 95% 구간이 0을 포함한다는 뜻이며, 두 값이 같다는 뜻은 아닙니다.
- 같은 평가 대상에서는 학습 데이터를 시간 순으로 고르든 무작위로 고르든 성능 차이가 작습니다. 최종 노트북 12-1절의 큰 차이(무작위 분할 전체 평가 AP 0.22 vs 시간 검증 0.11)는 **분할 방식보다 평가 대상(기간·구성)의 차이**에서 온 것으로 해석됩니다. 시간 순 검증의 의의는 성능 수치를 바꾸는 데보다, 선택에 미래 정보가 섞이지 않게 하고 운영 조건의 성능을 제시하는 데 있습니다.

**E3 단순 기준선**
- 무작위 선택 AP 0.024, 같은 검토량 Recall 0.332; 규칙 기반 (건강상태→나이) AP 0.091, 같은 검토량 Recall 0.756; 로지스틱 회귀 AP 0.102, 같은 검토량 Recall 0.777; 최종 모델 (XGBoost) AP 0.108, 같은 검토량 Recall 0.782.
- 차이 구간은 표의 '기준 대비 AP 차이'를 보세요 (기준 = 최종 모델).